<div style="text-align: justify; max-width: 90ch">

# Lecture 03d: Agentic RAG with LangGraph

**Status: Mandatory reading.**

`lec_03c` ended with a rule: a step that must always happen is not a decision, so write it in code. Here it shapes lecture 2's RAG system.
The search stays a fixed step; the model decides only how to word it (the rewrite `lec_02b` promised) and whether the chunks found are on topic.
[LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) (`lec_01d`) wires the steps into a loop, and the evaluation of `lec_03a`-`lec_03b` says whether it is worth its price.

**What this notebook covers**

- A question whose words miss the right notes.
- A model that rewrites the question, and one that grades the search.
- A LangGraph `StateGraph`: state, nodes, edges, a loop with a hard limit.
- The graph against the plain chain on four questions, and when a loop is worth it.

</div>

In [1]:
import os
from pathlib import Path
from typing import TypedDict

import mlflow
import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph

from course_helpers.agents import searches, step_table, total_tokens
from course_helpers.checks import check_mlflow
from course_helpers.corpus import load_corpus, load_eval_set
from course_helpers.evaluation import (
    DETERMINISTIC_SCORERS,
    per_question_table,
    build_eval_records_from,
)
from course_helpers.rag import (
    DEFAULT_RAG_PROMPT,
    LECTURE_KINDS,
    build_chroma_index,
    format_context_from,
    make_rag_chain,
    split_documents,
)

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-03-eval"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from course_helpers.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs.** The retriever and evaluation set of `lec_03a`, from the `course_helpers` helpers: the Chroma index of `lec_02d` (reused from `data/chroma/`,
built if missing), four chunks per question from the lecture notes and guides, and the evaluation set by question id. Then the chat model, thinking off, in Ollama's default window:
one window for rewrite, grade and answer, so the model never reloads (a judge added later shares it), with `num_predict=300`;
and the plain RAG chain on it.

</div>

In [3]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

documents = load_corpus(REPO / "corpus" / "uoa_py_course")
chunks = split_documents(documents, chunk_size=800, chunk_overlap=100)
store = build_chroma_index(
    chunks, "course_notes", DATA_DIR / "chroma", EMBED_MODEL, OLLAMA_HOST
)
retriever = store.as_retriever(
    search_kwargs={"k": 4, "filter": {"kind": {"$in": LECTURE_KINDS}}}
)

# No num_ctx: Ollama's default window for rewrite, grade, answer and any judge,
# so the model never reloads
llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=300,  # room for the source line; lec_03a's 200 can cut it off
)
rag_chain = make_rag_chain(retriever, llm)

eval_by_id = {
    row["id"]: row
    for row in load_eval_set(REPO / "corpus" / "eval" / "qa_eval_set.jsonl")
}

Ollama OK at http://localhost:11434; models ready: nomic-embed-text


Reusing 'course_notes': 2452 chunks in chroma/


<div style="text-align: justify; max-width: 90ch">

## 1. The student's words are not the notes' words

A student who has forgotten the word *tuple* asks: "Why can't I change an item of something written in round brackets?" The plain chain searches with the student's words,
so the search decides what the model reads. `retrieve_files_for` lists the files of the chunks the retriever returns. The table compares the student's wording with the same question in the notes' term,
and with q19 of `lec_03a`, whose answer is in `lec_04b`:

</div>

In [4]:
def retrieve_files_for(query):
    """File names of the chunks the retriever returns for a query, in rank order."""
    return [Path(d.metadata["source"]).name for d in retriever.invoke(query)]


STUDENT_QUESTION = "Why can't I change an item of something written in round brackets?"
EXPECTED_FILE = Path(eval_by_id["q19"]["expected_source"]).name
WORDINGS = {
    "student's words": STUDENT_QUESTION,
    "the notes' term": "Why can't I change an item of a tuple?",
    "q19": eval_by_id["q19"]["question"],
}
rows = []

for wording, text in WORDINGS.items():
    files = retrieve_files_for(text)
    rows.append(
        {
            "wording": wording,
            "expected file found": EXPECTED_FILE in files,
            "files": files,
        }
    )

pd.DataFrame(rows)

,wording,expected file found,files
0,student's words,False,"[lec_07a_pandas_df_index_slice.md, lec_07b_pan..."
1,the notes' term,True,"[lec_04b_index_slice_list_tuple_set.md, lec_04..."
2,q19,True,"[lec_04b_index_slice_list_tuple_set.md, lec_04..."


<div style="text-align: justify; max-width: 90ch">

Read the column `expected file found`. A search is not a sampled answer: with the same index and embedding model, the same text gives the same chunks on every run.
The student's words never say *tuple*, the word `lec_04b` uses. Where they miss `lec_04b` (the `False` row) and the notes' term finds it, one word made the difference on this index.
The fix is not a better answer from the wrong chunks but a search in the notes' words. Doing that automatically has names:

| Term | Meaning |
| --- | --- |
| **query rewriting** | a model rewrites the question into a search query in the notes' words |
| **grading** | a model judges whether the chunks found are on the question's topic |
| **agentic RAG** | RAG in which a model makes decisions about retrieval: what to search for, and whether to search again |

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. A model rewrites the question

Rewriting is a small task: one call, a short reply. The prompt asks for the terms the notes would use, and for the query only,
so the reply can go straight into the retriever. Thinking stays off, as in lecture 2.

</div>

In [5]:
REWRITE_PROMPT = ChatPromptTemplate.from_template(
    "Rewrite the question below as a short search query for the notes of a Python course, "
    "with the technical terms the notes would use. Reply with the query only."
    "\n\nQuestion: {question}"
)
rewrite_chain = REWRITE_PROMPT | llm | StrOutputParser()

query = rewrite_chain.invoke({"question": STUDENT_QUESTION})

print(query)

Why can't I change an item in a tuple using indexing?


<div style="text-align: justify; max-width: 90ch">

Read the query: does it use the notes' term, *tuple*? If so, the model supplied the word the student forgot. Whether the query finds `lec_04b` is in the next table;
in 6 of 6 example runs it did, but the query is a model's reply: read your own row.

## 3. A model grades the search

Some questions have no answer in the notes. Before answering, a second call **grades** the chunks: is the question's topic explained in them?
The verdict is a decision, `yes` or `no`, so it comes as structured output with an `enum` (`with_structured_output`, `lec_01d`).
The prompt asks about the *topic*, not the *answer*: in example runs a stricter wording said `no` to chunks that did answer, and every wrong `no` costs another search.

Three searches, each graded against its question: the student's, searched with the model's query and with its own words, and one the notes do not cover.

</div>

In [6]:
GRADE_PROMPT = ChatPromptTemplate.from_template(
    "Notes:\n{context}\n\nQuestion: {question}\n\n"
    "Is the question's topic explained in the notes above? Reply yes or no."
)
GRADE_SCHEMA = {
    "title": "Grade",
    "type": "object",
    "properties": {"topic_explained": {"type": "string", "enum": ["yes", "no"]}},
    "required": ["topic_explained"],
}
grade_chain = GRADE_PROMPT | llm.with_structured_output(
    GRADE_SCHEMA, method="json_schema"
)

OUT_OF_SCOPE_QUESTION = "Which lecture covers regular expressions?"
GRADE_CASES = [  # (case, question, text searched)
    ("model's query", STUDENT_QUESTION, query),
    ("student's words", STUDENT_QUESTION, STUDENT_QUESTION),
    ("not in the notes", OUT_OF_SCOPE_QUESTION, OUT_OF_SCOPE_QUESTION),
]
rows = []

for case, question, text in GRADE_CASES:
    found = retriever.invoke(text)
    verdict = grade_chain.invoke(
        {"context": format_context_from(found), "question": question}
    )
    files = [Path(d.metadata["source"]).name for d in found]
    rows.append(
        {
            "case": case,
            "expected file found": EXPECTED_FILE in files,
            "files": files,
            "grade": verdict["topic_explained"],
        }
    )

pd.DataFrame(rows)

,case,expected file found,files,grade
0,model's query,True,"[lec_04b_index_slice_list_tuple_set.md, lec_07...",yes
1,student's words,False,"[lec_07a_pandas_df_index_slice.md, lec_07b_pan...",yes
2,not in the notes,False,"[lec_11e_polynomial_regression.md, lec_11f_reg...",no


<div style="text-align: justify; max-width: 90ch">

Read `expected file found` in the first row: did the model's query reach `lec_04b`? Then read the grades:

- **Not in the notes**: no file covers regular expressions, and the grade should be `no`; in example runs (3 of 3) it was `no`.
- **Student's words**: a near miss, chunks from the course on another topic. The grade may say `yes` or `no` here; in example runs such in-domain near misses usually got `yes`.

So the grade tells on-topic from off-topic, not the right notebook from a near miss. That is why the rewrite runs first, on every question, not only after a `no`.
The grade always reads the student's question, never the query: a query can drift from what was asked.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. A graph: state, nodes, edges

The steps become a **graph** ([Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)). `create_agent` (`lec_03c`) built one for you;
building it yourself decides which steps the model controls.

| Term | Meaning |
| --- | --- |
| **state** | the dict every step reads and updates |
| **node** | one step: a function that receives the state and returns the keys it changes |
| **edge** | which node runs next |
| **conditional edge** | an edge chosen by a function of the state: where the loop is decided |
| `START`, `END` | where a run enters and leaves the graph |

Here the state holds the student's question, the model's query, the chunks found, the number of searches, the grade and the answer.

</div>

<div style="text-align: justify; max-width: 90ch">

Each node returns only the keys it changes; LangGraph merges them into the state. A key no node has written yet is missing, hence `state.get("searches", 0)`.

- **`rewrite`** writes the query; after a failed search, it gets the failed query and writes a different one.
- **`retrieve`** is code, not a decision (`lec_03c`): it searches on every pass and counts the searches.

</div>

In [7]:
RETRY_PROMPT = ChatPromptTemplate.from_template(
    "The search query below did not find the answer to the question. Write another short "
    "search query for the same question, in other words. "
    "Reply with the query only.\n\nQuestion: {question}\n\nQuery that failed: {query}"
)
retry_chain = RETRY_PROMPT | llm | StrOutputParser()


def rewrite(state):
    """Node: the model writes a search query; after a failed search, a different one."""
    if state.get("searches", 0) == 0:
        new_query = rewrite_chain.invoke({"question": state["question"]})
    else:
        new_query = retry_chain.invoke(
            {"question": state["question"], "query": state["query"]}
        )
    return {"query": new_query.strip()}


def retrieve(state):
    """Node: plain code. Search the notes with the model's query; count the search."""
    return {
        "documents": retriever.invoke(state["query"]),
        "searches": state.get("searches", 0) + 1,
    }

<div style="text-align: justify; max-width: 90ch">

- **`grade`** is the grade of section 3.
- **`answer`** uses the prompt of `lec_02d` (`DEFAULT_RAG_PROMPT`), with the chunks of the latest search and the student's question.

</div>

In [8]:
def grade(state):
    """Node: the model grades the chunks against the student's question."""
    verdict = grade_chain.invoke(
        {
            "context": format_context_from(state["documents"]),
            "question": state["question"],
        }
    )
    return {"grade": verdict["topic_explained"]}


answer_chain = (
    ChatPromptTemplate.from_template(DEFAULT_RAG_PROMPT) | llm | StrOutputParser()
)


def answer(state):
    """Node: the RAG answer, from the chunks of the latest search."""
    text = answer_chain.invoke(
        {
            "context": format_context_from(state["documents"]),
            "question": state["question"],
        }
    )
    return {"answer": text}

<div style="text-align: justify; max-width: 90ch">

The conditional edge is a function that returns the name of the next node. It holds the **hard limit**: after `MAX_SEARCHES` searches the graph answers with what it found, whatever the grade.
The answer prompt already says "if the notes do not contain the answer, say so", so a fruitless search ends in an honest refusal,
not a loop. The limit is code, so states written by hand test it without asking the model anything:

</div>

In [9]:
MAX_SEARCHES = 2


def route_after_grade(state):
    """Conditional edge: answer if the grade is yes or the search limit is reached."""
    if state["grade"] == "yes" or state["searches"] >= MAX_SEARCHES:
        return "answer"
    return "rewrite"


pd.DataFrame(
    [
        {
            "grade": g,
            "searches": n,
            "next node": route_after_grade({"grade": g, "searches": n}),
        }
        for g in ["yes", "no"]
        for n in [1, 2]
    ]
)

,grade,searches,next node
0,yes,1,answer
1,yes,2,answer
2,no,1,rewrite
3,no,2,answer


<div style="text-align: justify; max-width: 90ch">

Four rows, the same on every run: only a `no` after one search goes back to `rewrite`. Now wire the nodes. `StateGraph` needs the state's keys and types,
declared once with [`TypedDict`](https://docs.python.org/3/library/typing.html#typing.TypedDict): the type hints of `lec_03c`, for a dict.
`compile` checks the graph, so an edge to a node that does not exist fails here, not halfway through a run; it returns a Runnable with `invoke` and `stream`, like a chain:

</div>

In [10]:
RagState = TypedDict(
    "RagState",
    {
        "question": str,  # the student's question, never changed
        "query": str,  # the search query the model wrote
        "documents": list,  # the chunks of the latest search
        "searches": int,  # how many searches so far
        "grade": str,  # "yes" or "no"
        "answer": str,
    },
)

builder = StateGraph(RagState)
builder.add_node("rewrite", rewrite)
builder.add_node("retrieve", retrieve)
builder.add_node("grade", grade)
builder.add_node("answer", answer)

builder.add_edge(START, "rewrite")
builder.add_edge("rewrite", "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", route_after_grade, ["answer", "rewrite"])
builder.add_edge("answer", END)

graph = builder.compile()

pd.DataFrame(
    [
        {"from": e.source, "to": e.target, "conditional": e.conditional}
        for e in graph.get_graph().edges
    ]
)

,from,to,conditional
0,__start__,rewrite,False
1,grade,answer,True
2,grade,rewrite,True
3,retrieve,grade,False
4,rewrite,retrieve,False
5,answer,__end__,False


<div style="text-align: justify; max-width: 90ch">

Every edge you added, read back from the compiled graph; the two conditional ones leave `grade`. Drawn:

```text
START ──► rewrite ──► retrieve ──► grade ── yes, or searches = MAX_SEARCHES ──► answer ──► END
             ▲                       │
             └──────── no ───────────┘
```

Only two places are the model's: what to search for, and whether the search is on topic. The search itself and the limit are code.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. One run, step by step

`stream(..., stream_mode="updates")` yields what each node changed, as it runs ([streaming](https://docs.langchain.com/oss/python/langgraph/streaming)).
`step_table` from `course_helpers.agents` turns those updates into rows, with the files of the chunks instead of their text.
MLflow autolog (`lec_02d`) traces the run.

</div>

In [11]:
check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

MLflow OK at http://127.0.0.1:5010


In [12]:
student_updates = list(
    graph.stream({"question": STUDENT_QUESTION}, stream_mode="updates")
)

pd.DataFrame(step_table(student_updates)).fillna("")

,node,query,documents,searches,grade,answer
0,rewrite,Why can't I change an item in a tuple using in...,,,,
1,retrieve,,"lec_04b_in, lec_07b_pa, lec_04b_in, lec_04b_in",1,,
2,grade,,,,yes,
3,answer,,,,,The question is asking why we cannot change an...


<div style="text-align: justify; max-width: 90ch">

Read the rows top to bottom: the query, the files it found, the grade, and how many searches it took. The full answer:

</div>

In [13]:
print(student_updates[-1]["answer"]["answer"])

The question is asking why we cannot change an item of something written in round brackets (i.e., a tuple). 

From the notes, we know that tuples are **immutable**, meaning their elements cannot be changed after they are created. This is a fundamental property of tuples in Python.

### Explanation:
Tuples are **immutable** sequences, which means:
- You cannot modify existing elements.
- You cannot add, remove, or change elements once the tuple is created.

### Source File:
[source: lecture_04/lec_04b_index_slice_list_tuple_set.md]

### Answer:
Tuples are immutable, meaning their elements cannot be changed after they are created. This is why you cannot change an item of something written in round brackets.


<div style="text-align: justify; max-width: 90ch">

Does it say that round brackets make a tuple, and that a list can change while a tuple cannot? Does it name `lec_04b`? The student never wrote *tuple*:
if the rewrite's query supplied it (section 2), the search with it reached the notes that explain it.

### ⏸ Pause point

Open the newest trace in the MLflow UI (experiment `llm-course-03-eval`): each node is a span, with a `RETRIEVER` span inside `retrieve` and a `CHAT_MODEL` span for every model call.
How many model calls did this answer take, and how many would the plain chain have made?

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. The limit at work

The regular expressions question of section 3 has no answer in the notes. If every search grades `no`, the loop runs until the limit stops it:

</div>

In [14]:
out_of_scope_updates = list(
    graph.stream({"question": OUT_OF_SCOPE_QUESTION}, stream_mode="updates")
)

pd.DataFrame(step_table(out_of_scope_updates)).fillna("")

,node,query,documents,searches,grade,answer
0,rewrite,Which lecture covers regular expressions?,,,,
1,retrieve,,"lec_11e_po, lec_11f_re, lec_12d_lo, lec_12d_lo",1,,
2,grade,,,,no,
3,rewrite,Which lecture covers regular expressions in th...,,,,
4,retrieve,,"lec_11e_po, lec_11f_re, lec_12d_lo, lec_12d_lo",2,,
5,grade,,,,no,
6,answer,,,,,The notes provided do not mention anything abo...


<div style="text-align: justify; max-width: 90ch">

Count the `retrieve` rows and read the second query. In example runs (3 of 3) there were two searches, each graded `no`,
and the answer said the notes do not cover regular expressions. Had a grade said `yes`, the graph would have answered after one search;
either way it stops, because the limit is code. Without it, a grade that always says `no` would loop until LangGraph's own `recursion_limit` (`lec_03c`) stopped the run,
and its default is about ten thousand steps: in practice, nothing would stop the loop.

</div>

<div style="text-align: justify; max-width: 90ch">

## 7. The graph against the chain

Is the loop worth it? The method of `lec_03b`, on four fixed questions: q19 and q15 as the evaluation set words them, and **reworded** as a student might ask,
section 1's question and a vague q15. A reworded question keeps the expectations of its original, so every scorer still applies.

The scorers of `lec_03a`, plus two trace scorers from `course_helpers.agents` for the price:

- **`searches`**: how many `RETRIEVER` spans, that is searches, the answer took.
- **`total_tokens`**: every token read and written by every model call of the answer, rewrite and grade included.

No judge here, to keep the run short; adding `lec_03b`'s is Exercise 3.7 of the lecture 3 exercises. The four questions, then one evaluation run per system, the chain first:

</div>

In [15]:
REWORDED = {
    "q19": STUDENT_QUESTION,
    "q15": "I built a little web demo of my model, how can my classmates open it from their own laptops?",
}
questions = [eval_by_id["q19"], eval_by_id["q15"]] + [
    {**eval_by_id[qid], "id": qid + "r", "question": text}
    for qid, text in REWORDED.items()
]
records = build_eval_records_from(questions)
SCORERS = [*DETERMINISTIC_SCORERS, searches, total_tokens]

pd.DataFrame(questions)[["id", "question", "expected_source"]]

,id,question,expected_source
0,q19,What is the difference between a list and a tu...,lecture_04/lec_04b_index_slice_list_tuple_set.md
1,q15,Where can I deploy a Gradio app so that anyone...,lecture_09/lec_09c_gradio_app_huggingface_depl...
2,q19r,Why can't I change an item of something writte...,lecture_04/lec_04b_index_slice_list_tuple_set.md
3,q15r,"I built a little web demo of my model, how can...",lecture_09/lec_09c_gradio_app_huggingface_depl...


In [16]:
def predict_chain(question):
    """The plain RAG chain: search with the question as written."""
    return rag_chain.invoke(question)


results_by_system = {}

with mlflow.start_run(run_name="plain chain"):
    results_by_system["plain chain"] = mlflow.genai.evaluate(
        data=records, predict_fn=predict_chain, scorers=SCORERS
    )

2026/10/11 17:13:27 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


2026/10/11 17:13:27 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


<div style="text-align: justify; max-width: 90ch">

The same records and scorers for the graph, in a run of its own:

</div>

In [17]:
def predict_graph(question):
    """The graph: rewrite, search, grade, perhaps search again, answer."""
    return graph.invoke({"question": question})["answer"]


with mlflow.start_run(run_name="agentic graph"):
    results_by_system["agentic graph"] = mlflow.genai.evaluate(
        data=records, predict_fn=predict_graph, scorers=SCORERS
    )

2026/10/11 17:13:33 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


<div style="text-align: justify; max-width: 90ch">

One row per question and system, with the columns that tell the two apart, plus `seconds`: the time each answer took (`execution_duration` of the result table, as in `lec_03b`'s `compare_variants`):

</div>

In [18]:
COLUMNS = [
    "retrieval_hit",
    "refused",
    "cites_expected_source",
    "searches",
    "total_tokens",
]
ID_BY_QUESTION = {row["question"]: row["id"] for row in questions}


def build_system_table_from(results):
    """The COLUMNS of per_question_table, plus the seconds each answer took."""
    frame = results.result_df
    # "request" is the question, or a dict holding it, depending on the trace
    ids = [
        ID_BY_QUESTION[r["question"] if isinstance(r, dict) else r]
        for r in frame["request"]
    ]
    # execution_duration is in milliseconds
    seconds = pd.Series(frame["execution_duration"].to_numpy() / 1000, index=ids)
    table = per_question_table(results, questions, SCORERS)[COLUMNS]
    return table.assign(seconds=seconds.round(1))


pd.concat(
    {
        name: build_system_table_from(results)
        for name, results in results_by_system.items()
    },
    names=["system", "id"],
).swaplevel().sort_index(level="id", sort_remaining=False)

retrieval_hit  refused  cites_expected_source  searches  \
id   system                                                                   
q15  plain chain             True    False                   True       1.0   
     agentic graph           True    False                   True       1.0   
q15r plain chain            False     True                  False       1.0   
     agentic graph          False     True                  False       1.0   
q19  plain chain             True    False                   True       1.0   
     agentic graph           True    False                   True       1.0   
q19r plain chain            False    False                  False       1.0   
     agentic graph           True    False                   True       1.0   

                    total_tokens  seconds  
id   system                                
q15  plain chain           808.0      1.9  
     agentic graph        1612.0      1.8  
q15r plain chain           926.0      1.3  
     agentic graph        1584.0      2.8  
q19  plain chain           654.0      4.6  
     agentic graph        1361.0      3.3  
q19r plain chain          1303.0      4.2  
     agentic graph        1259.0      4.5

<div style="text-align: justify; max-width: 90ch">

Compare the rows of each id, chain against graph:

- **q19, q15**: both systems should find the expected file; look at what the graph pays for the same hit in `total_tokens`, and compare `seconds` (see the note on waiting below).
- **q19r, q15r, chain**: on a given index the chain's search repeats from run to run; read its `retrieval_hit`. Where it is `False`, does the answer refuse or use the wrong notes?
- **q19r, q15r, graph**: q19r's rewrite found `lec_04b` in 6 of 6 example runs; q15r's rewrite varies, and found its file in most example runs but not all.
  A may-or-may-not case next to one that does not depend on luck: read your own rows.

`retrieval_hit` reads the first search, the one after the rewrite. Did the graph ever search twice? Where the graph searched once, its gain came from the rewrite alone.
`seconds` includes waiting, as MLflow sends several questions at once: compare, don't read values.

</div>

<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

In the MLflow UI, select the two runs and compare them. Would a rewrite ever lose a file that the student's own words found?
What would catch it?

</div>

<div style="text-align: justify; max-width: 90ch">

## 8. When is an agentic loop worth it?

The decision, recorded like those of `lec_01b` and `lec_01d`:

- **Decision:** a rewrite step where students ask in their own words; the grade-and-retry loop only where a second search can find something new.
- **Requirements:** the right notes in the prompt; an answer, or an honest "not in the notes"; tokens and seconds per answer.
- **Evidence:** sections 1, 6 and 7, as example runs on four questions.
- **Worth it when:** users' words differ from the documents'; a retry has somewhere new to look (another source, a relaxed filter, a tool).
- **Not worth it when:** the questions already match the documents; latency matters most; nobody evaluates the loop.
- **Revisit when:** the questions, the documents or the model change.

The graph's prompts, its builder (`make_agentic_rag(retriever, llm, max_searches)`), `step_table` and the scorers `searches` and `total_tokens` are in `course_helpers.agents`.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A search finds what the question's words find: a question in other words can miss the right notes, on every run.
- **Query rewriting**: a model writes the search in the notes' words. **Grading**: a model judges whether the chunks are on topic.
- A **`StateGraph`** has a **state** (a `TypedDict`), **nodes** (functions returning the keys they change) and **edges**;
  a **conditional edge** decides the loop.
- The search is code, the decisions are the model's, and a **hard limit** in code ends every loop.
- An agentic loop is a variant: evaluate it against the plain chain, quality against tokens and seconds.

**Next: the lecture 3 exercises**; optionally `lec_03e`, how far to trust the judge, and `lec_03f`, letting DSPy optimize the prompt.

</div>